# ArcGIS Feature Service CSV Exporter

This notebook downloads **all records** from an Esri ArcGIS Feature Service
endpoint and saves them as a `.csv` file to your computer.

**How to use:**
1. Paste your ArcGIS service URL in Step 1 below
2. Click **Runtime > Run all** in the menu bar (or press Ctrl+F9)
3. Wait for it to finish — the CSV file will automatically download to your computer

---
## Step 1: Paste your ArcGIS URL here

Replace the example URL below with your own. The URL should end with a layer
number like `/FeatureServer/0` or `/MapServer/3`.

In [ ]:
#------------------------------------------------------------
# PASTE YOUR URL BETWEEN THE QUOTES BELOW
#------------------------------------------------------------
ARCGIS_URL = "https://sampleserver6.arcgisonline.com/arcgis/rest/services/Census/MapServer/3"

#------------------------------------------------------------
# OPTIONAL: Set a custom output filename (leave blank for auto)
#------------------------------------------------------------
OUTPUT_FILENAME = ""

---
## Step 2: Run the export

Click **Runtime > Run all** above, or just click the play button on each cell
in order. The cells below do all the work automatically.

In [ ]:
import csv
import json
import time
import urllib.parse
import urllib.request
from datetime import datetime, timezone


def make_request(url, params=None, max_retries=3):
    """Make an HTTP GET request with retry logic."""
    if params:
        query_string = urllib.parse.urlencode(params)
        full_url = f"{url}?{query_string}"
    else:
        full_url = url

    for attempt in range(max_retries):
        try:
            req = urllib.request.Request(full_url, headers={
                "User-Agent": "ArcGIS-CSV-Exporter/1.0",
                "Accept": "application/json",
            })
            with urllib.request.urlopen(req, timeout=60) as response:
                body = response.read().decode("utf-8")
                return json.loads(body)
        except Exception as e:
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"  Request failed ({e}), retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise RuntimeError(
                    f"Request failed after {max_retries} attempts: {e}"
                ) from e


def get_layer_metadata(base_url):
    """Fetch layer metadata including fields and maxRecordCount."""
    print(f"Fetching layer metadata from:\n  {base_url}")
    metadata = make_request(base_url, {"f": "json"})
    if "error" in metadata:
        code = metadata["error"].get("code", "")
        msg = metadata["error"].get("message", "Unknown error")
        raise RuntimeError(f"ArcGIS API error {code}: {msg}")
    return metadata


def get_total_count(base_url):
    """Get the total number of records in the layer."""
    result = make_request(base_url + "/query", {
        "where": "1=1",
        "returnCountOnly": "true",
        "f": "json",
    })
    if "error" in result:
        raise RuntimeError(f"Count query failed: {result['error']}")
    return result.get("count", 0)


def get_all_object_ids(base_url):
    """Retrieve all object IDs from the layer."""
    result = make_request(base_url + "/query", {
        "where": "1=1",
        "returnIdsOnly": "true",
        "f": "json",
    })
    if "error" in result:
        raise RuntimeError(f"OID query failed: {result['error']}")
    oid_field = result.get("objectIdFieldName", "OBJECTID")
    oids = sorted(result.get("objectIds", []))
    return oid_field, oids


def fetch_features_paginated(base_url, max_record_count):
    """Fetch all features using resultOffset pagination."""
    all_features = []
    offset = 0
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "returnGeometry": "true",
            "f": "json",
            "resultOffset": offset,
            "resultRecordCount": max_record_count,
        }
        result = make_request(base_url + "/query", params)
        if "error" in result:
            raise RuntimeError(f"Query failed at offset {offset}: {result['error']}")
        features = result.get("features", [])
        all_features.extend(features)
        print(f"  Fetched {len(all_features)} records so far...")
        if not result.get("exceededTransferLimit", False):
            break
        offset += max_record_count
    return all_features


def fetch_features_by_oids(base_url, oid_field, oids, batch_size):
    """Fetch all features by batching object IDs (fallback method)."""
    all_features = []
    for i in range(0, len(oids), batch_size):
        batch = oids[i : i + batch_size]
        params = {
            "objectIds": ",".join(str(oid) for oid in batch),
            "outFields": "*",
            "returnGeometry": "true",
            "f": "json",
        }
        result = make_request(base_url + "/query", params)
        if "error" in result:
            raise RuntimeError(
                f"OID batch query failed (batch starting at index {i}): "
                f"{result['error']}"
            )
        features = result.get("features", [])
        all_features.extend(features)
        print(f"  Fetched {len(all_features)} records so far...")
    return all_features


def format_field_value(value, field_type):
    """Format a field value based on its ArcGIS field type."""
    if value is None:
        return ""
    if field_type in ("esriFieldTypeDate",):
        try:
            dt = datetime.fromtimestamp(value / 1000, tz=timezone.utc)
            return dt.strftime("%Y-%m-%d %H:%M:%S")
        except (OSError, ValueError, TypeError):
            return str(value)
    return value


def geometry_to_string(geometry):
    """Convert an ArcGIS geometry object to a readable string."""
    if geometry is None:
        return ""
    if "x" in geometry and "y" in geometry:
        x, y = geometry["x"], geometry["y"]
        if x == "NaN" or y == "NaN":
            return ""
        return f"POINT ({x} {y})"
    if "paths" in geometry:
        parts = []
        for path in geometry["paths"]:
            coords = ", ".join(f"{p[0]} {p[1]}" for p in path)
            parts.append(f"({coords})")
        return f"MULTILINESTRING ({', '.join(parts)})"
    if "rings" in geometry:
        parts = []
        for ring in geometry["rings"]:
            coords = ", ".join(f"{p[0]} {p[1]}" for p in ring)
            parts.append(f"({coords})")
        return f"POLYGON ({', '.join(parts)})"
    if "points" in geometry:
        coords = ", ".join(f"({p[0]} {p[1]})" for p in geometry["points"])
        return f"MULTIPOINT ({coords})"
    return json.dumps(geometry)


def normalize_url(url):
    """Clean up the input URL."""
    url = url.strip().rstrip("/")
    url = url.split("?")[0]
    if url.endswith("/query"):
        url = url[: -len("/query")]
    return url


def generate_output_filename(metadata):
    """Generate a default output filename from the layer name."""
    name = metadata.get("name", "arcgis_export")
    safe_name = "".join(c if c.isalnum() or c in (" ", "-", "_") else "_" for c in name)
    safe_name = safe_name.strip().replace(" ", "_")
    return f"{safe_name}.csv"


print("Functions loaded successfully.")

In [ ]:
from google.colab import files

# Validate input
if not ARCGIS_URL or ARCGIS_URL.strip() == "":
    raise ValueError("Please paste your ArcGIS URL in the Step 1 cell above and re-run.")

base_url = normalize_url(ARCGIS_URL)
print("ArcGIS Feature Service CSV Exporter")
print("=" * 40)

# --- Get layer metadata ---
metadata = get_layer_metadata(base_url)
layer_name = metadata.get("name", "Unknown")
geometry_type = metadata.get("geometryType")
max_record_count = metadata.get("maxRecordCount", 1000)
field_list = metadata.get("fields", [])

if not field_list:
    raise RuntimeError("No fields found in layer metadata. Check your URL.")

supports_pagination = (
    metadata.get("advancedQueryCapabilities", {})
    .get("supportsPagination", False)
)

print(f"\nLayer name:          {layer_name}")
print(f"Geometry type:       {geometry_type or 'None (table)'}")
print(f"Fields:              {len(field_list)}")
print(f"Max records/query:   {max_record_count}")
print(f"Supports pagination: {supports_pagination}")

# --- Count records ---
total_count = get_total_count(base_url)
print(f"Total records:       {total_count}")

if total_count == 0:
    print("\nNo records to export.")
else:
    # --- Download all features ---
    print(f"\nDownloading records...")
    if supports_pagination:
        all_features = fetch_features_paginated(base_url, max_record_count)
    else:
        print("  Pagination not supported, using Object ID batching...")
        oid_field, oids = get_all_object_ids(base_url)
        all_features = fetch_features_by_oids(base_url, oid_field, oids, max_record_count)

    print(f"\nTotal features retrieved: {len(all_features)}")

    # --- Write CSV ---
    output_path = OUTPUT_FILENAME.strip() if OUTPUT_FILENAME.strip() else generate_output_filename(metadata)
    if not output_path.endswith(".csv"):
        output_path += ".csv"

    field_map = {f["name"]: f.get("type", "") for f in field_list}
    field_names = [f["name"] for f in field_list]
    has_geometry = geometry_type is not None and geometry_type != ""

    headers = list(field_names)
    if has_geometry:
        headers.append("GEOMETRY")

    with open(output_path, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(headers)
        for feature in all_features:
            attrs = feature.get("attributes", {})
            row = []
            for name in field_names:
                raw = attrs.get(name)
                ftype = field_map.get(name, "")
                row.append(format_field_value(raw, ftype))
            if has_geometry:
                row.append(geometry_to_string(feature.get("geometry")))
            writer.writerow(row)

    print(f"\nExported {len(all_features)} records to {output_path}")
    print("Downloading to your computer now...")
    files.download(output_path)

---
## Troubleshooting

| Problem | Solution |
| --- | --- |
| **"No fields found"** | Double-check your URL ends with a layer number like `/FeatureServer/0` |
| **"Request failed"** | The service may be down or require authentication (only public services are supported) |
| **CSV didn't download** | Check your browser's download bar — it may have been blocked by a popup blocker |
| **Want to run again with a different URL** | Change the URL in Step 1, then click **Runtime > Run all** again |